In [1]:
!pip install -q opencv-python-headless requests openpyxl tqdm
print("Install sucessful")
import json

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 23.3 MB/s eta 0:00:00
Install sucessful


In [2]:
USERNAME = "octopus"
PASSWORD = "communication42"
VIDEO_FOLDER = "https://repo.octopus-intelligence.org/public/O-vulgaris-Nity-2026-2-20--/Right%20Top/Local/"
OCTOPUS_NAME = "Nity"
OUTPUT_FILE = "/content/octopus_ethogram.xlsx"

In [3]:
JSON_PATH = "/small-json.json"

with open(JSON_PATH, "r") as f:
    event_data = json.load(f)

In [4]:
DATES_TO_RUN = sorted(set(
    entry["Date"].split(" ")[0]
    for entry in event_data["Nity events"]
    if "Date" in entry and entry["Date"]
))

print(f"Loaded {len(DATES_TO_RUN)} unique date(s) from JSON:")
for d in DATES_TO_RUN:
    print(f"•{d}")

Loaded 105 unique date(s) from JSON:
•2025-09-17
•2025-09-18
•2025-09-19
•2025-09-21
•2025-09-23
•2025-09-27
•2025-09-29
•2025-10-01
•2025-10-02
•2025-10-03
•2025-10-04
•2025-10-05
•2025-10-06
•2025-10-07
•2025-10-08
•2025-10-09
•2025-10-10
•2025-10-11
•2025-10-13
•2025-10-14
•2025-10-16
•2025-10-17
•2025-10-19
•2025-10-20
•2025-10-22
•2025-10-23
•2025-10-28
•2025-10-30
•2025-11-01
•2025-11-02
•2025-11-05
•2025-11-07
•2025-11-08
•2025-11-09
•2025-11-10
•2025-11-11
•2025-11-12
•2025-11-13
•2025-11-14
•2025-11-15
•2025-11-16
•2025-11-17
•2025-11-18
•2025-11-19
•2025-11-20
•2025-11-21
•2025-11-22
•2025-11-23
•2025-11-24
•2025-11-25
•2025-11-27
•2025-11-28
•2025-11-30
•2025-12-01
•2025-12-02
•2025-12-03
•2025-12-05
•2025-12-06
•2025-12-07
•2025-12-08
•2025-12-10
•2025-12-11
•2025-12-12
•2025-12-13
•2025-12-14
•2025-12-16
•2025-12-17
•2025-12-18
•2025-12-19
•2025-12-20
•2025-12-22
•2025-12-23
•2025-12-24
•2025-12-25
•2025-12-26
•2025-12-27
•2025-12-28
•2025-12-29
•2025-12-30
•2025-12-31
•20

In [5]:
STOP_AFTER_SECONDS = 8
MIN_MOTION = 0.0026
CHECK_EVERY_N_FRAMES = 8
PIXEL_SENSITIVITY = 27
PROCESS_WIDTH = 320   # Resize frames to this width before analysis (None = no resize)
print("Settings saved")


Settings saved


In [6]:
import re, os
import cv2
import numpy as np
import requests
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import timedelta
from urllib.parse import quote, urlparse, urlunparse, unquote
from tqdm.notebook import tqdm
from pathlib import Path
print("sucessful")

sucessful


In [7]:
def to_timestamp(seconds):
    return str(timedelta(seconds=int(seconds)))

In [8]:
def to_duration(seconds):
    seconds = int(seconds)
    m, s = divmod(seconds, 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

In [9]:
def add_auth_to_url(url):
    p = urlparse(url)
    netloc = f"{quote(USERNAME)}:{quote(PASSWORD)}@{p.hostname}"
    return urlunparse(p._replace(netloc=netloc))


In [10]:
def get_all_videos():
    print("Finding videos...")
    session = requests.Session()
    session.auth = (USERNAME, PASSWORD)
    # Get all date folders actually present in the online repo
    response = session.get(VIDEO_FOLDER)
    repo_date_folders = set(
        d.strip("/")
        for d in re.findall(r'href="(\d{4}-\d{2}-\d{2}/)"', response.text)
    )
    print(f"   Repo contains {len(repo_date_folders)} date folder(s) online")
    # Cross-check: JSON dates vs what's actually in the repo
    matched_dates  = sorted(d for d in DATES_TO_RUN if d in repo_date_folders)
    skipped_dates  = sorted(d for d in DATES_TO_RUN if d not in repo_date_folders)
    if skipped_dates:
        print(f"\n{len(skipped_dates)} date(s) from JSON not found in repo — skipping:")
        for d in skipped_dates:
            print(f"Rejected :{d}")
    print(f"\n{len(matched_dates)} date(s) matched and will be processed:")
    for d in matched_dates:
        print(f"accepted:{d}")
    if not matched_dates:
        return []
    # Collect video files only for matched dates
    all_videos = []
    for date_str in matched_dates:
        folder_url = VIDEO_FOLDER + date_str + "/"
        response = session.get(folder_url)
        mp4_files = re.findall(r'href="([^"]+\.mp4)"', response.text)
        if not mp4_files:
            print(f"No .mp4 files found in {date_str} — skipping")
            continue
        for mp4 in sorted(mp4_files):
            all_videos.append({
                "date"    : date_str,
                "filename": unquote(mp4),
                "url"     : folder_url + mp4,
            })
    print(f"\nFound {len(all_videos)} video(s) total across matched dates")
    return all_videos

In [11]:
def find_motion(video_url, video_name):
    # Open video stream (no download needed)
    stream_url = add_auth_to_url(video_url)
    cap = cv2.VideoCapture(stream_url)
    if not cap.isOpened():
        print(f"Could not open video: {video_name}")
        return []
    fps          = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration     = total_frames / fps
    still_needed = int(STOP_AFTER_SECONDS * fps / CHECK_EVERY_N_FRAMES)
    segments    = []        # list of (start_sec, end_sec)
    moving      = False     # is octopus currently moving?
    move_start  = 0.0       # when did it start moving?
    still_count = 0         # how many still frames in a row?
    prev_frame  = None
    frame_num   = 0
    # Progress bar for this video
    bar = tqdm(total=int(duration), unit="s",
               desc=f"  {video_name[:35]}", leave=False)
    last_sec = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        frame_num += 1
        current_sec = frame_num / fps
        # Update progress bar every second
        if current_sec - last_sec >= 1:
            bar.update(int(current_sec - last_sec))
            last_sec = current_sec
        # Only analyse every Nth frame
        if frame_num % CHECK_EVERY_N_FRAMES != 0:
            continue
        # Resize frame for faster processing
        if PROCESS_WIDTH is not None:
            h, w = frame.shape[:2]
            scale = PROCESS_WIDTH / w
            new_h = int(h * scale)
            frame = cv2.resize(frame, (PROCESS_WIDTH, new_h), interpolation=cv2.INTER_LINEAR)

        # Convert to grayscale and blur (reduces noise)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)

        if prev_frame is None:
            prev_frame = gray
            continue
        # Compare to previous frame
        diff = cv2.absdiff(prev_frame, gray)
        changed_pixels = np.sum(diff > PIXEL_SENSITIVITY)
        fraction_changed = changed_pixels / diff.size
        is_moving = fraction_changed >= MIN_MOTION

        # Track motion start/stop
        if is_moving:
            if not moving:
                move_start = current_sec   # octopus started moving
                moving = True
            still_count = 0
        else:  # not moving
            if moving:
                still_count += 1
                if still_count >= still_needed:
                    end_sec = current_sec - STOP_AFTER_SECONDS
                    if end_sec > move_start:
                        segments.append((move_start, end_sec))
                        bar.write(f"    ✦ [{len(segments):02d}]  "
                                  f"{to_timestamp(move_start)} → {to_timestamp(end_sec)}"
                                  f"  ({to_duration(end_sec - move_start)})")
                    moving = False
                    still_count = 0

        prev_frame = gray
    if moving:
        end_sec = frame_num / fps
        if end_sec > move_start:
            segments.append((move_start, end_sec))

    bar.update(int(duration) - int(last_sec))
    bar.close()
    cap.release()
    return segments


In [12]:
def save_excel(rows, filepath):
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title ="Ethogram"

    headers = ["Octopus", "Date", "Video File", "Start Time", "End Time", "Duration"]
    for col, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.font = Font(name="Arial", bold=True, color="000000", size=11)
        cell.fill = PatternFill("solid", fgColor="006400")
        cell.alignment = Alignment(horizontal="center")

    for i, row in enumerate(rows, 2):
        color = "EFF3FB" if i % 2 == 0 else "FFFFFF"
        values = [row["octopus"], row["date"], row["filename"],
                  row["start"], row["end"], row["duration"]]
        for col, val in enumerate(values, 1):
            cell = ws.cell(row=i, column=col, value=val)
            cell.fill = PatternFill("solid", fgColor=color)
            cell.font = Font(name="Arial", size=10)
            cell.alignment = Alignment(horizontal="center" if col > 1 else "left")

    for col, width in enumerate([18, 12, 30, 12, 12, 12], 1):
        ws.column_dimensions[get_column_letter(col)].width = width

    ws.freeze_panes = "A2"
    wb.save(filepath)



In [13]:
videos = get_all_videos()

Finding videos...
   Repo contains 52 date folder(s) online

96 date(s) from JSON not found in repo — skipping:
Rejected :2025-09-17
Rejected :2025-09-18
Rejected :2025-09-19
Rejected :2025-09-21
Rejected :2025-09-23
Rejected :2025-09-27
Rejected :2025-09-29
Rejected :2025-10-01
Rejected :2025-10-02
Rejected :2025-10-03
Rejected :2025-10-04
Rejected :2025-10-05
Rejected :2025-10-06
Rejected :2025-10-07
Rejected :2025-10-08
Rejected :2025-10-09
Rejected :2025-10-10
Rejected :2025-10-11
Rejected :2025-10-13
Rejected :2025-10-14
Rejected :2025-10-16
Rejected :2025-10-17
Rejected :2025-10-19
Rejected :2025-10-20
Rejected :2025-10-22
Rejected :2025-10-23
Rejected :2025-10-28
Rejected :2025-10-30
Rejected :2025-11-01
Rejected :2025-11-02
Rejected :2025-11-05
Rejected :2025-11-07
Rejected :2025-11-08
Rejected :2025-11-09
Rejected :2025-11-10
Rejected :2025-11-11
Rejected :2025-11-12
Rejected :2025-11-13
Rejected :2025-11-14
Rejected :2025-11-15
Rejected :2025-11-16
Rejected :2025-11-17
Reject

In [14]:
if not videos:
    print(" No videos found! Check your DATES_TO_RUN or VIDEO_FOLDER.")
else:
    all_rows = []
    overall_bar = tqdm(videos, desc="Overall", unit="video")

    for video in overall_bar:
        label = f"{video['date']} | {video['filename']}"
        overall_bar.set_postfix_str(label[:45])
        tqdm.write(f"\n▶  {label}")
        segments = find_motion(video["url"], video["filename"])
        tqdm.write(f"   → {len(segments)} segment(s) found")
        if not segments:
            all_rows.append({
                "octopus" : OCTOPUS_NAME,
                "date"    : video["date"],
                "filename": video["filename"],
                "start"   : "—", "end": "—", "duration": "—"
            })
        else:
            for start, end in segments:
                all_rows.append({
                    "octopus" : OCTOPUS_NAME,
                    "date"    : video["date"],
                    "filename": video["filename"],
                    "start"   : to_timestamp(start),
                    "end"     : to_timestamp(end),
                    "duration": to_duration(end - start),
                })
        save_excel(all_rows, OUTPUT_FILE)
        tqdm.write(f" Saved — {len(all_rows)} rows so far")

    overall_bar.close()
    print(f"\nAll done! {len(all_rows)} rows saved to:")
    print(f"   {OUTPUT_FILE}")

Overall:   0%|          | 0/432 [00:00<?, ?video/s]


▶  2026-02-22 | 000000--vv-1.mp4


  000000--vv-1.mp4:   0%|          | 0/1797 [00:00<?, ?s/s]

    ✦ [01]  0:00:01 → 0:00:09  (00:08)
    ✦ [02]  0:00:18 → 0:00:55  (00:36)
    ✦ [03]  0:01:07 → 0:01:31  (00:24)
    ✦ [04]  0:01:41 → 0:02:03  (00:22)
    ✦ [05]  0:02:12 → 0:02:21  (00:09)
    ✦ [06]  0:02:31 → 0:03:11  (00:40)
    ✦ [07]  0:03:31 → 0:03:38  (00:07)
    ✦ [08]  0:03:50 → 0:04:17  (00:27)
    ✦ [09]  0:04:25 → 0:04:27  (00:01)
    ✦ [10]  0:04:36 → 0:04:40  (00:04)
    ✦ [11]  0:04:57 → 0:06:19  (01:22)
    ✦ [12]  0:06:28 → 0:06:38  (00:10)
    ✦ [13]  0:06:50 → 0:07:06  (00:15)
    ✦ [14]  0:07:15 → 0:07:33  (00:17)
    ✦ [15]  0:07:54 → 0:08:31  (00:36)
    ✦ [16]  0:08:42 → 0:08:59  (00:16)
    ✦ [17]  0:09:16 → 0:09:45  (00:28)
    ✦ [18]  0:09:58 → 0:10:18  (00:19)
    ✦ [19]  0:10:40 → 0:10:58  (00:17)
    ✦ [20]  0:11:07 → 0:11:07  (00:00)
    ✦ [21]  0:11:16 → 0:11:40  (00:23)
    ✦ [22]  0:11:50 → 0:12:07  (00:17)
    ✦ [23]  0:12:17 → 0:12:21  (00:04)
    ✦ [24]  0:12:31 → 0:12:41  (00:09)
    ✦ [25]  0:12:52 → 0:12:59  (00:07)
    ✦ [26]  0:13:08 → 0:1

  003000--vv-1.mp4:   0%|          | 0/1796 [00:00<?, ?s/s]

    ✦ [01]  0:00:08 → 0:00:25  (00:16)
    ✦ [02]  0:00:36 → 0:00:52  (00:15)
    ✦ [03]  0:01:02 → 0:01:07  (00:04)
    ✦ [04]  0:01:16 → 0:01:17  (00:00)
    ✦ [05]  0:01:27 → 0:03:08  (01:41)
    ✦ [06]  0:03:17 → 0:03:24  (00:07)
    ✦ [07]  0:03:38 → 0:03:47  (00:09)
    ✦ [08]  0:03:58 → 0:04:16  (00:18)
    ✦ [09]  0:04:29 → 0:04:57  (00:27)
    ✦ [10]  0:05:10 → 0:07:05  (01:55)
    ✦ [11]  0:07:17 → 0:08:33  (01:15)
    ✦ [12]  0:08:46 → 0:09:37  (00:50)
    ✦ [13]  0:09:47 → 0:11:30  (01:42)
    ✦ [14]  0:11:39 → 0:11:49  (00:09)
    ✦ [15]  0:12:04 → 0:12:12  (00:08)
    ✦ [16]  0:12:24 → 0:13:38  (01:14)
    ✦ [17]  0:13:49 → 0:14:38  (00:48)
    ✦ [18]  0:14:55 → 0:15:00  (00:04)
    ✦ [19]  0:15:10 → 0:15:21  (00:10)
    ✦ [20]  0:15:39 → 0:16:34  (00:54)
    ✦ [21]  0:16:43 → 0:17:59  (01:15)
    ✦ [22]  0:18:09 → 0:18:09  (00:00)
    ✦ [23]  0:18:18 → 0:18:26  (00:08)


KeyboardInterrupt: 